In [1]:
import torch

In [ ]:
# deepseek版本

def build_rope_cache(seq_len, dim):
    """
    构建cos, sin 缓存
    seq_len是序列长度
    dim是旋转维度，必须是偶数，一般等于head_dim
    """

    # 假设 dim=64, 则freq 有dim//2 = 32个频率
    half_dim = dim // 2
    inv_dreq = 1.0/(10000**(torch.arange(0,half_dim).float()/half_dim))

    position = torch.aragne(seq_len).float()
    freqs = torch.einsum("i,j->ij", position, inv_dreq)

    cos_cached = torch.cos(freqs)
    sin_cached = torch.sin(freqs)

    return cos_cached, sin_cached

def apply_rope(x,cos, sin):
    """
    x :[batch, seq_len, dim]

    cos, sin : [seq_len, dim/2]
    """

    b,n,d = x.shape
    half = d//2
    x1 = x[:,:, :half]
    x2 = x[:,:, half:]
    cos = cos[:n,:].unsqueeze(0)
    sin = sin[:n,:].unsqueeze(0)

    x_rotated = torch.cat([x1*cos-x2*sin, x1*sin+x2*cos],dim = -1)

    return x_rotated





In [2]:
import torch.nn as nn
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# rethinkfun 视频介绍版本

# 旋转位置编码 每个element是 xcos(theta) 或者xsin(theta)
# 定义theta
# 定义position, 和f_i
#f_i 是频率 ， position是位置
# 得到两个完整的cos, sin矩阵
# contheta , sinthetha
# -sintheta, costhetha
# (-x2, x1) 
# costheta *x + sinthetha * rotated x


f_i = 1/100000 **(torch.arange(0,d_model,2).float()/ d_model)
# 这里是d/2

position = torch.arange( seq_len).float().unsqueeze(0).expand(batch_size, seq_len)

# x 维度 batch_size, seq_len, d_model
# q,k 维度 batch_size, seq_len, num_heads, head_dim

# position * f_i得到的维度应该是 跟x维度一样的 

inv_freq = f_i[None, :, None].float().expand(position.size(0), -1, 1)
positions =position[:,None, :].float()

freq_s = (inv_freq @ positions).transpose(1,2)
# 这里已经把dim/2中 中间，转移到最后一维

# 有了freqs，现在开始计算 
embed = torch.cat((freq_s, freq_s), dim = -1)
cos = embed.cos()
sin = embed.sin()

def rotate(x,d_model):
    x1 = x[..., :d_model//2]
    x2 = x[..., d_model//2:]
    return torch.cat((-x2,x1), dim=-1)

def apply_rope(q,k ,d_model):
    # 注意这里如果q,k是[batch, seq_len, num_heads, head_dim]，则需要先reshape
    # cos = cos.unsqueeze(1)
    # sin = sin.unsqueeze(1)

    q = cos*q + sin*rotate(q, d_model)
    k = cos*k + sin*rotate(k, d_model)
    return q, k




In [ ]:
# 完整的 按照我的逻辑的rope
class RoPE(nn.Module):
    def __init__(self, seq_len, dim):
        super(RoPE, self).__init__()
        assert dim % 2 == 0, "dim必须是偶数，因为RoPE每两个维度一组旋转"
        self.seq_len = seq_len
        self.dim = dim
        inv_freq = 1.0/(10000 ** (torch.arange(0,dim, 2).float() /dim ))

        self.register_buffer("inv_freq", inv_freq, persistent=False)

        # 在这里可以预先定义固定的位置编码旋转矩阵
        # 在这里定义好是因为这个矩阵是固定的，不需要每次都计算，避免重复计算
        position_ids = torch.arange(seq_len).float()
        freqs = torch.einsum("i,j->ij", position_ids , inv_freq ) # (seq_len, dim/2)
        # 等价于 torch.outer(position_ids, inv_freq)
        embed = torch.cat((freqs,freqs),dim=-1) # # [max_seq_len, dim]
        self.register_buffer("cos_cached", embed.cos(), persistent=False)
        self.register_buffer("sin_cached", embed.sin(), persistent=False)

    def forward(self, q,k,position_ids):
        """
        q, k: [batch, seq_len, num_heads, head_dim]
        """
        b, n, h, d = q.shape
        # 鉴于目前cos, sin都是(seq_len, dim）
        cos = self.cos_cached[position_ids].unsqueeze(0).unsqueeze(2)
        sin = self.sin_cached[position_ids].unsqueeze(0).unsqueeze(2)

       
        embed_q = cos * q + sin *  self.rotate_half(q)
        embed_k = cos * k + sin *  self.rotate_half(k)
        return embed_q, embed_k

    @staticmethod
    def rotate_half(x):
        x1 = x[..., :x.shape[-1]//2]
        x2 = x[..., x.shape[-1]//2:]
        return torch.cat((-x2, x1),dim =-1) 

